In [39]:
# =============================================================================
# CÓDIGO INDICADORES MENSUALES MULTIANUALES (MEDIAS, MÁXIMOS Y MÍNIMOS)
# V.1.0 - Asignación automática de abreviaturas a los nombres de archivo
# Realizado por I.C Ricardo Santiago Guerrero - SIMAC
# =============================================================================
## CARGA DE LIBRERÍAS
import pandas as pd
import os
import glob

# Ruta de la carpeta principal
folder_path = r"C:\Users\ricar\OneDrive\Almacen\Trabajo\5. SIMAC\10. Series\2026-04"
ruta_salida = r"C:\Users\ricar\OneDrive\Almacen\Trabajo\5. SIMAC\9. Indicadores_climaticos\4.Revision_de_series\6.Medios maximos multianuales\4.Resultados"

# Ruta del archivo de localización en la misma carpeta del .ipynb
ruta_localizacion = os.path.join(os.getcwd(), "Datos_localizacion_estaciones.xlsx")

# Variable opcional para saltar las primeras filas
filas_a_saltar = 3 

# Variable auxiliar para seleccionar una pequeña muestra.
muestra_archivos = 200

# Variable para definir la profundidad de búsqueda de carpetas
niveles_carpetas = 3

# Variable para definir qué columna se va a analizar
variable_a_analizar = "Precipitación [mm]" #Se deben introducir de la siguiente manera: Temperatura [°C],Precipitación [mm],Nivel Corregido [cm], Velocidad del Viento,Dirección del Viento,Dirección de la Rosa,Presión Barométrica [mmHg],Humedad Relativa [%],Radiación Solar [W/m2],Evapotranspiración

# =============================================================================
# IDENTIFICACIÓN AUTOMÁTICA DE LA ABREVIATURA
# =============================================================================
var_lower = variable_a_analizar
if "Precipitación [mm]" in var_lower:
    abreviatura = "PPT"
elif "Temperatura [°C]" in var_lower:
    abreviatura = "TEMP"
elif "Nivel Corregido [cm]" in var_lower:
    abreviatura = "NIV"
elif "Velocidad del Viento" in var_lower:
    abreviatura = "VV"
elif "Dirección del Viento" in var_lower and "rosa" not in var_lower:
    abreviatura = "DV"
elif "Dirección de la Rosa" in var_lower:
    abreviatura = "DROSA"
elif "Presión Barométrica [mmHg]" in var_lower:
    abreviatura = "PRES"
elif "Humedad Relativa [%]" in var_lower:
    abreviatura = "HR"
elif "Radiación Solar [W/m2]" in var_lower:
    abreviatura = "RAD"
elif "Evapotranspiración" in var_lower:
    abreviatura = "ETP"
else:
    abreviatura = "VAR"

print(f"Variable identificada: {variable_a_analizar} -> Sufijo a utilizar: _{abreviatura}")
print("-" * 60)

# =============================================================================
# EXTRACCIÓN Y CÁLCULO DE INDICADORES MENSUALES MULTIANUALES
# =============================================================================
csv_files = []
for nivel in range(niveles_carpetas):
    subcarpetas = ["*"] * nivel
    file_pattern = os.path.join(folder_path, *subcarpetas, "*.csv")
    csv_files.extend(glob.glob(file_pattern))

if 'muestra_archivos' in locals():
    csv_files = csv_files[:muestra_archivos]

print(f"Se van a procesar {len(csv_files)} archivos CSV...")
print("-" * 60)

meses = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
         7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}

# Listas separadas para cada indicador
resultados_medias = []
resultados_maximos = []
resultados_minimos = []

for archivo in csv_files:
    try:
        # 1. Procesar el nombre del archivo
        nombre_archivo = os.path.basename(archivo)
        
        if len(nombre_archivo) > 11:
            codigo_estacion = nombre_archivo[:9].strip()
            nombre_estacion = nombre_archivo[11:].replace('.csv', '').replace('_corregido', '').strip()
        else:
            codigo_estacion = "N/A"
            nombre_estacion = nombre_archivo.replace('.csv', '').replace('_corregido', '').strip()

        # 2. Verificar la existencia de las columnas y leer
        try:
            encabezados = pd.read_csv(archivo, skiprows=filas_a_saltar, nrows=0).columns
            if variable_a_analizar in encabezados and 'Fecha' in encabezados:
                print(f"✓ ÉXITO: Columna detectada en {nombre_archivo}")
                df = pd.read_csv(archivo, skiprows=filas_a_saltar, usecols=['Fecha', variable_a_analizar])
            else:
                print(f"✗ AVISO: Columna no detectada en {nombre_archivo}. Saltando...")
                continue
        except Exception as e:
            print(f"✗ ERROR al leer {nombre_archivo}: {e}")
            continue
            
        df.columns = ['Fecha', 'Valor']
        df['Fecha'] = pd.to_datetime(df['Fecha'], format='%d/%m/%Y', errors='coerce')
        df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')
        df = df.dropna(subset=['Fecha', 'Valor'])

        # 3. DOBLE AGREGACIÓN (Mensual y luego Multianual)
        df['Año'] = df['Fecha'].dt.year
        df['Mes'] = df['Fecha'].dt.month
        
        # Paso A: Agrupamos por Año y Mes. 
        if "precipitaci" in variable_a_analizar.lower():
            datos_mensuales = df.groupby(['Año', 'Mes'])['Valor'].sum().reset_index()
        else:
            datos_mensuales = df.groupby(['Año', 'Mes'])['Valor'].mean().reset_index()
            
        # Paso B: Calculamos media, máximo y mínimo multianual
        medias_multianuales = datos_mensuales.groupby('Mes')['Valor'].mean().round(2)
        maximos_multianuales = datos_mensuales.groupby('Mes')['Valor'].max().round(2)
        minimos_multianuales = datos_mensuales.groupby('Mes')['Valor'].min().round(2)

        # 4. Estructurar las filas para esta estación
        fila_med = {'Nombre de la estacion': nombre_estacion, 'Codigo de la estacion': codigo_estacion}
        fila_max = {'Nombre de la estacion': nombre_estacion, 'Codigo de la estacion': codigo_estacion}
        fila_min = {'Nombre de la estacion': nombre_estacion, 'Codigo de la estacion': codigo_estacion}

        for num_mes, nombre_mes in meses.items():
            fila_med[nombre_mes] = medias_multianuales.get(num_mes, None)
            fila_max[nombre_mes] = maximos_multianuales.get(num_mes, None)
            fila_min[nombre_mes] = minimos_multianuales.get(num_mes, None)

        resultados_medias.append(fila_med)
        resultados_maximos.append(fila_max)
        resultados_minimos.append(fila_min)

    except Exception as e:
        print(f"Error procesando {nombre_archivo}: {e}")

# =============================================================================
# 5. INTEGRACIÓN CON DATOS DE LOCALIZACIÓN Y EXPORTACIÓN MÚLTIPLE
# =============================================================================

# Diccionario con los nombres de archivo adaptados con la abreviatura
archivos_a_generar = {
    f"Medias_Mensuales_Multianuales_{abreviatura}.xlsx": resultados_medias,
    f"Maximos_Mensuales_Multianuales_{abreviatura}.xlsx": resultados_maximos,
    f"Minimos_Mensuales_Multianuales_{abreviatura}.xlsx": resultados_minimos
}

# Intentar cargar el archivo de localización
df_loc = None
columnas_localizacion = []
try:
    if os.path.exists(ruta_localizacion):
        df_loc = pd.read_excel(ruta_localizacion)
        columnas_localizacion = [col for col in df_loc.columns if col != 'Nombre actual estación']
        print("\n✓ Datos de localización listos para integración.")
    else:
        print(f"\n✗ AVISO: No se encontró el archivo de localización en:\n{ruta_localizacion}")
except Exception as e:
    print(f"\n✗ ERROR al leer los datos de localización: {e}")

print("-" * 60)

# Diccionario para almacenar los DataFrames finales para el Excel consolidado
dfs_consolidados = {}

# Procesar y exportar cada uno de los 3 archivos individuales
for nombre_archivo, datos_resultado in archivos_a_generar.items():
    if datos_resultado:
        df_final = pd.DataFrame(datos_resultado)
        
        if df_loc is not None:
            df_final = pd.merge(df_final, df_loc, left_on='Nombre de la estacion', right_on='Nombre actual estación', how='left')
            if 'Nombre actual estación' in df_final.columns:
                df_final = df_final.drop(columns=['Nombre actual estación'])
                
        orden_columnas = ['Nombre de la estacion', 'Codigo de la estacion'] + columnas_localizacion + list(meses.values())
        
        orden_final = [col for col in orden_columnas if col in df_final.columns]
        df_final = df_final[orden_final]
        
        # Guardar DataFrame para el archivo consolidado
        nombre_hoja = nombre_archivo.replace(".xlsx", "")
        dfs_consolidados[nombre_hoja] = df_final
        
        # Exportar el archivo individual
        ruta_archivo_salida = os.path.join(ruta_salida, nombre_archivo)
        df_final.to_excel(ruta_archivo_salida, index=False)
        
        print(f"¡Generado individual! -> {nombre_archivo}")
    else:
        print(f"No hay datos para generar {nombre_archivo}.")

# =============================================================================
# 6. EXPORTACIÓN DEL ARCHIVO CONSOLIDADO
# =============================================================================
if dfs_consolidados:
    nombre_consolidado = f"Indicadores_Mensuales_Multianuales_{abreviatura}.xlsx"
    ruta_consolidado = os.path.join(ruta_salida, nombre_consolidado)
    
    with pd.ExcelWriter(ruta_consolidado, engine='openpyxl') as writer:
        for nombre_hoja, df_hoja in dfs_consolidados.items():
            # Openpyxl tiene un límite de 31 caracteres para los nombres de las hojas.
            # Cortamos el nombre si es necesario para evitar errores.
            nombre_hoja_corto = nombre_hoja[:31] 
            df_hoja.to_excel(writer, sheet_name=nombre_hoja_corto, index=False)
            
    print("-" * 60)
    print(f"¡Generado consolidado! -> {nombre_consolidado}")
    print(f"\nTodos los archivos se guardaron exitosamente en:\n{ruta_salida}")

Variable identificada: Precipitación [mm] -> Sufijo a utilizar: _PPT
------------------------------------------------------------
Se van a procesar 124 archivos CSV...
------------------------------------------------------------
✓ ÉXITO: Columna detectada en 001 - 001 - Q. Palogrande - Ruta 30_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 015 - Q. Olivares - El Popal_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 016 - Q. Tesorito_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 026 - Q. El Guamo - CDI San Sebastián_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 027 - Q. Guayabal - Recinto del Pensamiento_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 028 - Q. Manizales - Maltería_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 029 - Q. Olivares - Bocatoma_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 030 - Q. Olivares - Aguas de Manizales_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 031 - Q. Manzanares - IBC_corregido.csv
✓ ÉXITO: Columna detectada en 001 - 03

In [38]:
# =============================================================================
# MÓDULO DE VERIFICACIÓN: SERIE HISTÓRICA COMPLETA DE UN MES
# =============================================================================

# 1. DEFINE TUS VARIABLES DE BÚSQUEDA AQUÍ:
estacion_a_revisar = "873 - 007" 
mes_a_revisar = 1 

# =============================================================================
archivo_encontrado = None
for archivo in csv_files:
    if estacion_a_revisar in os.path.basename(archivo):
        archivo_encontrado = archivo
        break

if archivo_encontrado:
    print(f"✓ Archivo: {os.path.basename(archivo_encontrado)}")
    print("-" * 60)
    
    df_rev = pd.read_csv(archivo_encontrado, skiprows=filas_a_saltar, usecols=['Fecha', variable_a_analizar])
    df_rev.columns = ['Fecha', 'Valor']
    df_rev['Fecha'] = pd.to_datetime(df_rev['Fecha'], format='%d/%m/%Y', errors='coerce')
    df_rev['Valor'] = pd.to_numeric(df_rev['Valor'], errors='coerce')
    df_rev = df_rev.dropna(subset=['Fecha', 'Valor'])
    
    # Filtramos el mes
    df_mes = df_rev[df_rev['Fecha'].dt.month == mes_a_revisar].copy()
    
    if not df_mes.empty:
        df_mes['Año'] = df_mes['Fecha'].dt.year
        
        # Agregamos los datos del mes por cada año
        if "precipitaci" in variable_a_analizar.lower():
            totales_por_anio = df_mes.groupby('Año')['Valor'].sum()
            tipo_calc = "Suma mensual"
        else:
            totales_por_anio = df_mes.groupby('Año')['Valor'].mean()
            tipo_calc = "Promedio mensual"
            
        media_multianual = totales_por_anio.mean()
        
        print(f"--- SERIE HISTÓRICA PARA EL MES DE {meses[mes_a_revisar].upper()} ---")
        print(f"Operación aplicada por año: {tipo_calc}\n")
        
        # Imprimir todos los años encontrados en forma de tabla
        print(" Año  |   Valor")
        print("-" * 20)
        for anio, valor in totales_por_anio.items():
            print(f"{anio}  |  {valor:.4f}")
            
        print("-" * 60)
        print(f"▶ MEDIA MENSUAL MULTIANUAL CALCULADA: {media_multianual:.6f}")
        print(f"(Este es el valor que debe estar exportado en el Excel)")
        
    else:
        print(f"No hay datos para {meses[mes_a_revisar]}.")
else:
    print(f"✗ No se encontró la estación '{estacion_a_revisar}'.")

✓ Archivo: 873 - 007 - Villamaría - Hospital_corregido.csv
------------------------------------------------------------
--- SERIE HISTÓRICA PARA EL MES DE ENERO ---
Operación aplicada por año: Promedio mensual

 Año  |   Valor
--------------------
2011  |  19.1343
2013  |  20.1181
2014  |  18.9511
2015  |  20.0168
2016  |  21.6358
2017  |  19.4539
2018  |  19.6140
2019  |  21.9749
2020  |  20.5981
2022  |  23.5567
2024  |  20.1880
2025  |  19.1889
2026  |  17.2210
------------------------------------------------------------
▶ MEDIA MENSUAL MULTIANUAL CALCULADA: 20.127040
(Este es el valor que debe estar exportado en el Excel)
